# Bridge Infrastructure Analytics: Predicting Bridge Deck Deterioration Using PySpark, Machine Learning, SQL, and Power BI

## Project Overview

Bridges are essential components of transportation infrastructure, and timely maintenance is critical for ensuring public safety and minimizing repair costs. Environmental factors such as freeze–thaw cycles significantly contribute to bridge deck deterioration, making predictive analysis valuable for infrastructure management.

This project develops a machine learning framework using PySpark and Databricks to analyze large-scale bridge inspection data collected from multiple U.S. states between 2015 and 2025. The study investigates how bridge age, environmental exposure, structural characteristics, and traffic conditions influence bridge deck deterioration.

Multiple machine learning models, including Linear Regression, Gradient-Boosted Trees, and XGBoost, are trained and compared to predict bridge deck condition. The project also incorporates MongoDB for NoSQL analytics and will include an interactive Power BI dashboard for infrastructure decision support.

---

## Project Objectives

- Analyze bridge deterioration using large-scale inspection data.
- Study the impact of freeze–thaw exposure on bridge deck condition.
- Engineer predictive features for bridge deterioration.
- Compare multiple machine learning models.
- Identify the most influential factors affecting bridge health.
- Support infrastructure maintenance through data-driven insights.

# Dataset Description

## Data Source

The dataset consists of bridge inspection records obtained from the National Bridge Inventory. For this study, bridge inspection data from four U.S. states covering the period from **2015 to 2025** were analyzed.

The dataset contains structural, environmental, geographic, and traffic-related attributes describing bridge characteristics and inspection outcomes.

---

## Key Features

The dataset includes variables such as:

- Bridge age
- State and county
- Average daily traffic
- Main-span material
- Structural design type
- Deck area
- Bridge length
- Number of spans
- Freeze–thaw classification
- Environmental exposure
- Deck condition rating

---

## Analytics and Machine Learning Goals

### Data Analytics

- Compare bridge deterioration across states and freeze–thaw categories.
- Analyze the relationship between bridge age, traffic, material, design, and deck condition.
- Identify infrastructure patterns using PySpark, SQL, MongoDB, and Power BI.
- Develop visual summaries to support maintenance planning.

### Machine Learning

- Predict the bridge deck condition rating.
- Engineer structural, environmental, and traffic-related features.
- Compare Linear Regression, Gradient-Boosted Trees, and XGBoost.
- Evaluate models using RMSE and R².
- Identify the most influential predictors of bridge deterioration.

---

## Decision-Support Objective

The analysis is intended to help transportation and infrastructure stakeholders identify deterioration patterns and prioritize bridges for further inspection, maintenance, or rehabilitation.

# Data Loading and Initial Inspection

## Purpose

The first step of the workflow is to load the bridge inspection dataset into a PySpark DataFrame. PySpark enables distributed processing of large infrastructure datasets, making it suitable for scalable data cleaning, exploratory analysis, and machine learning.

During this stage, the dataset schema, record count, and sample observations are examined to verify that the data has been imported correctly before further preprocessing.

In [ ]:
# ============================================================
# DATA LOADING
# ============================================================
### Environment Note

#This notebook was developed and executed in the Databricks environment using Apache Spark (PySpark). The `spark` session is automatically initialized by Databricks. To run this notebook, upload the `PS1.csv` dataset to the configured Databricks Volume:

#`/Volumes/workspace/default/raw-data/PS1.csv`

# Import the National Bridge Inventory dataset into a Spark DataFrame

file_path = "/Volumes/workspace/default/raw-data/PS1.csv"

df = (
    spark.read
    .option("header", True)          # file has header row
    .option("inferSchema", True)     # detect column types
    .option("multiLine", True)       # allows records spanning multiple lines
    .option("quote", "'")            # your file uses single quotes for quoting
    .option("escape", "'")           # escape using single quote
    .option("sep", ",")              # comma delimiter
    .csv(file_path)
)

# Display the first five records
df.show(5)
# Display dataset schema
df.printSchema()

# Display total number of bridge inspection records
print("Total records:", df.count())


NameError: name 'spark' is not defined

# Data Cleaning and Preprocessing

## Purpose

Bridge inspection datasets contain a large number of attributes, many of which are not required for this analysis. To improve computational efficiency and model performance, only the relevant variables are selected.

During preprocessing, the dataset is cleaned, important columns are renamed for better readability, and the data is prepared for exploratory analysis and machine learning.

In [0]:
# ============================================================
# DATA CLEANING AND PREPROCESSING
# ============================================================

# Select relevant bridge inspection variables and rename columns
# to improve readability for analysis and machine learning.

from pyspark.sql import functions as F

# Preserve the original DataFrame before preprocessing
raw_df = df

bridges_df = (
    raw_df
    .select(
        F.col("Year0").alias("year"),                       # Year of record
        F.col("11").alias("state_code"),                    # State numeric code
        F.col("-2").alias("state_name"),                    # State name
        F.col("State3").alias("structure_id_seq"),          # Internal structure id (numeric)
        F.col("Code4").alias("structure_number_raw"),       # Structure number (string with junk)
        F.col("15").alias("owner_agency"),                  # Owner agency text
        F.col("-6").alias("owner_agency_code"),             # Owner agency code
        F.col("State7").alias("county_name"),               # County name
        F.col("Name8").alias("year_built"),                 # Year built
        F.col("8").alias("main_span_material"),             # Main span material
        F.col("-10").alias("main_span_design"),             # Main span design
        F.col("Structure11").alias("deck_area_sqft"),       # Deck area (sq ft)
        F.col("Number12").alias("deck_condition_rating"),   # Deck condition rating
        F.col("2025").alias("latitude"),                    # Latitude
        F.col("NBI").alias("longitude"),                    # Longitude
        F.col("Structure15").alias("adt"),                  # Average Daily Traffic
        F.col("Number16").alias("bridge_age"),              # Bridge age (years)
        F.col("22").alias("freeze_thaw_cycles")             # Number of freeze–thaw cycles
    )
)

# Display sample records after selecting and renaming columns
bridges_df.show(5, truncate=False)

# Display updated schema
bridges_df.printSchema()

print("Total bridge inspection records:", bridges_df.count())

# Detailed Data Cleaning

## Purpose

This stage standardizes identifier, text, and condition-rating fields before analysis and modeling.

The cleaning process:

- extracts numeric bridge structure identifiers,
- trims and standardizes text-based fields,
- converts `"NULL"` and empty strings into proper null values,
- extracts numeric deck-condition ratings,
- prepares the cleaned variables for type conversion and downstream analysis.

In [0]:
# ============================================================
# DETAILED DATA CLEANING
# ============================================================

from pyspark.sql import functions as F

clean_df = (
    bridges_df

    # 1) Extract numeric characters from the raw structure number
    .withColumn(
        "structure_number",
        F.regexp_replace("structure_number_raw", r"[^0-9]", "")
    )

    # 2) Standardize and trim text-based fields
    .withColumn("owner_agency",       F.trim(F.regexp_replace("owner_agency",       ",", " ")))
    .withColumn("county_name",        F.trim(F.regexp_replace("county_name",        ",", " ")))
    .withColumn("main_span_material", F.trim(F.regexp_replace("main_span_material", ",", " ")))
    .withColumn("main_span_design",   F.trim(F.regexp_replace("main_span_design",   ",", " ")))

    # 3) Convert placeholder values and empty strings to null
    .withColumn("deck_condition_rating_clean",
                F.when((F.col("deck_condition_rating") == "NULL") |
                       (F.col("deck_condition_rating") == "") |
                       (F.col("deck_condition_rating").isNull()), None)
                 .otherwise(F.col("deck_condition_rating")))

    # 4) Extract digits from the deck-condition rating
    .withColumn("deck_condition_rating_digits",
                F.regexp_extract("deck_condition_rating_clean", r"([0-9]+)", 1))

    # 5) Convert empty extracted values to null
    .withColumn("deck_condition_rating_digits",
                F.when(F.col("deck_condition_rating_digits") == "", None)
                 .otherwise(F.col("deck_condition_rating_digits")))

    # 6) FINAL: Safe cast using try_cast()
    .withColumn("deck_condition_rating_int",
                F.expr("try_cast(deck_condition_rating_digits AS INT)"))
)

# Clean final output: Drop intermediate columns
clean_df = clean_df.drop("deck_condition_rating",
                         "deck_condition_rating_clean",
                         "deck_condition_rating_digits"
                        ) \
                   .withColumnRenamed("deck_condition_rating_int",
                                      "deck_condition_rating")

# Preview
clean_df.select(
    "year", "state_name", "county_name",
    "structure_number",
    "deck_condition_rating",
    "deck_area_sqft",
    "latitude", "longitude",
    "adt", "bridge_age", "freeze_thaw_cycles"
).show(10, truncate=False)

clean_df.printSchema()


In [0]:
# ============================================================
# DATA QUALITY ASSESSMENT
# ============================================================

from pyspark.sql import functions as F

# 1) Count NULLs per column
null_counts = clean_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in clean_df.columns
])

print("🔍 NULL counts per column:")
null_counts.show(truncate=False)

# 2) Basic descriptive stats for numeric columns
numeric_cols = ["deck_area_sqft", "latitude", "longitude",
                "adt", "bridge_age", "freeze_thaw_cycles",
                "deck_condition_rating"]

print("📊 Basic statistics:")
clean_df.select(numeric_cols).describe().show()

# 3) Distribution of deck condition rating
print("📈 Deck Condition Rating Distribution:")
clean_df.groupBy("deck_condition_rating").count().orderBy("deck_condition_rating").show()

# 4) Distribution of freeze-thaw cycles
print("❄️ Freeze-Thaw Cycles Distribution:")
clean_df.groupBy("freeze_thaw_cycles").count().orderBy("freeze_thaw_cycles").show(20)


In [0]:
# -------------------------------------------------------
# FILTER OUT INVALID / IMPOSSIBLE VALUES
# -------------------------------------------------------

from pyspark.sql import functions as F

filtered_df = (
    clean_df
    .filter((F.col("latitude") >= 24) & (F.col("latitude") <= 50))          # valid US lat
    .filter((F.col("longitude") >= -125) & (F.col("longitude") <= -66))     # valid US long
    .filter(F.col("bridge_age") > 0)                                        # no negative age
    .filter(F.col("deck_area_sqft") < 200000)                               # remove huge outliers
    .filter(F.col("deck_condition_rating").isNotNull())                     # deck condition required
    .filter(F.col("freeze_thaw_cycles").isNotNull())                        # freeze-thaw required
)

print("Original rows:", clean_df.count())
print("Filtered rows:", filtered_df.count())


Exploratory Data Analysis (EDA)

In [0]:
# ------------------------------------------
#  EDA

# 1. COMPUTE CORRELATION MATRIX IN SPARK
# ------------------------------------------
import pyspark.sql.functions as F

num_cols = [
    "deck_condition_rating",
    "freeze_thaw_cycles",
    "bridge_age",
    "adt",
    "deck_area_sqft"
]

corr_values = {}

for c1 in num_cols:
    corr_values[c1] = []
    for c2 in num_cols:
        val = filtered_df.stat.corr(c1, c2)
        corr_values[c1].append(val)

# Convert to pandas dataframe for heatmap
import pandas as pd

corr_df = pd.DataFrame(corr_values, index=num_cols)
corr_df



# ------------------------------------------
# 2. CORRELATION HEATMAP (MATPLOTLIB)
# ------------------------------------------
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.imshow(corr_df, cmap='coolwarm', interpolation='nearest')
plt.colorbar()

plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha='right')
plt.yticks(range(len(num_cols)), num_cols)

plt.title("Correlation Heatmap")
plt.show()



In [0]:
#Deck Condition vs Freeze–Thaw Bucketization

#Freeze–thaw cycles will show more meaningful patterns when binned, not raw.

from pyspark.sql.functions import when

# Create buckets
binned_df = filtered_df.withColumn(
    "ft_zone",
    when(F.col("freeze_thaw_cycles") < 20, "Low (0-20)")
    .when(F.col("freeze_thaw_cycles") < 50, "Moderate (20-50)")
    .when(F.col("freeze_thaw_cycles") < 80, "High (50-80)")
    .otherwise("Extreme (80+)")
)

display(
    binned_df.groupBy("ft_zone")
        .agg(F.avg("deck_condition_rating").alias("avg_rating"),
             F.count("*").alias("count"))
        .orderBy("ft_zone")
)

In [0]:
#Deck Condition by State

#Some states manage winter damage better.

display(
    filtered_df.groupBy("state_name")
        .agg(F.avg("deck_condition_rating").alias("avg_rating"),
             F.avg("freeze_thaw_cycles").alias("avg_ft"))
        .orderBy("avg_rating")
)

In [0]:
#Deck Condition vs ADT (Traffic)

#Traffic effects are often clearer when bucketed.

adt_df = filtered_df.withColumn(
    "adt_level",
    when(F.col("adt") < 500, "Very Low")
    .when(F.col("adt") < 2000, "Low")
    .when(F.col("adt") < 10000, "Medium")
    .when(F.col("adt") < 30000, "High")
    .otherwise("Very High")
)

display(
    adt_df.groupBy("adt_level")
        .agg(F.avg("deck_condition_rating").alias("avg_rating"),
             F.count("*").alias("count"))
        .orderBy("adt_level")
)

In [0]:
#Deck Condition Distribution by Material

#Material plays a major role in deterioration and freeze–thaw damage:

display(
    filtered_df.groupBy("main_span_material")
        .agg(F.avg("deck_condition_rating").alias("avg_rating"),
             F.count("*").alias("count"))
        .orderBy("avg_rating")
)

In [0]:
#Deck Condition vs Design Type

#Design determines how decks are reinforced.

display(
    filtered_df.groupBy("main_span_design")
        .agg(F.avg("deck_condition_rating").alias("avg_rating"),
             F.count("*").alias("count"))
        .orderBy("avg_rating")
)

## Groupwise Analysis

### Purpose

Groupwise analysis is performed to examine how bridge condition varies across different operational and environmental categories. By aggregating inspection records, this analysis identifies patterns between deck condition ratings and factors such as freeze–thaw cycles.

The results provide insights into how environmental exposure influences bridge deterioration and help identify groups that may require higher maintenance priority.

In [0]:
#Groupwise Analysis

# Calculate the average deck condition rating for each freeze–thaw cycle category
filtered_df.groupBy("freeze_thaw_cycles") \
    .agg(F.avg("deck_condition_rating").alias("avg_deck_rating"),
         F.count("*").alias("count")) \
    .orderBy("freeze_thaw_cycles") \
    .show(20)

In [0]:
# Calculate the average freeze–thaw cycle exposure for each deck condition rating
filtered_df.groupBy("deck_condition_rating") \
    .agg(F.avg("freeze_thaw_cycles").alias("avg_freeze_thaw"),
         F.count("*").alias("count")) \
    .orderBy("deck_condition_rating") \
    .show()

### Observation

The aggregated results reveal how bridge deck condition changes across different freeze–thaw cycle categories. Higher freeze–thaw exposure may be associated with lower deck condition ratings, indicating the influence of environmental stress on bridge deterioration.

These findings provide useful insights for infrastructure maintenance planning and risk prioritization.

## Basic Trend Analysis

### Purpose

This section explores overall trends between bridge characteristics and deck condition ratings. Aggregated statistics help identify whether factors such as bridge age and geographic location are associated with infrastructure deterioration.

These trends provide valuable insights for maintenance planning and long-term asset management.

In [0]:
# Calculate the average deck condition rating for bridges of different ages
filtered_df.groupBy("bridge_age") \
    .agg(F.avg("deck_condition_rating").alias("avg_rating")) \
    .orderBy("bridge_age") \
    .show(20)

In [0]:
# Compare average deck condition and freeze–thaw exposure across states
filtered_df.groupBy("state_name") \
    .agg(F.avg("deck_condition_rating").alias("avg_rating"),
         F.avg("freeze_thaw_cycles").alias("avg_ft")) \
    .orderBy("avg_rating") \
    .show(10)

## Visualization

In [0]:
# Visualize the relationship between freeze–thaw cycles and average deck condition rating
ft_trend = (
    filtered_df.groupBy("freeze_thaw_cycles")
        .agg(F.avg("deck_condition_rating").alias("avg_deck_rating"))
        .orderBy("freeze_thaw_cycles")
)

display(ft_trend)

In [0]:
#Deck Condition vs Bridge Age
age_trend = (
    filtered_df.groupBy("bridge_age")
        .agg(F.avg("deck_condition_rating").alias("avg_rating"))
        .orderBy("bridge_age")
)

display(age_trend)

In [0]:
#Distribution of Deck Condition
display(
    filtered_df.select("deck_condition_rating")
)

In [0]:
#Distribution of Freeze–Thaw Cycles
display(
    filtered_df.select("freeze_thaw_cycles")
)

In [0]:
# Convert Spark DF → Pandas
age_pdf = filtered_df.select("bridge_age").toPandas()

import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

plt.hist(
    age_pdf["bridge_age"],
    bins=30,                # same bin count
    edgecolor="black",      # ← adds dark lines between bars
    linewidth=1.0           # ← thickness of the lines
)

plt.title("Distribution of Bridge Age", fontsize=16)
plt.xlabel("Bridge Age (years)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(axis='y', alpha=0.3)

plt.show()


In [0]:
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

# States you want to compare
states = ["Alabama", "Minnesota", "Virginia", "Wisconsin"]

# COLUMN TO PLOT  (change if needed)
value_column = "adt"     # deck_condition_rating if your values are 0–9

# Convert Spark → pandas
df = (
    filtered_df.filter(F.col("state_name").isin(states))
               .select("state_name", value_column)
               .dropna()
               .toPandas()
)

# Prepare data for boxplot
data = [df[df["state_name"] == s][value_column] for s in states]

# Color palette
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

# Plot
plt.figure(figsize=(10, 6))

box = plt.boxplot(
    data,
    labels=states,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    flierprops=dict(marker="o", markerfacecolor="black", markersize=5, alpha=0.5)
)

# Apply colors to each box
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

plt.title("Deck Condition by State", fontsize=14)
plt.xlabel("State")
plt.ylabel("Deck Condition Rating")   # change label if using ADT
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [0]:
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Create freeze–thaw class in Spark
ft_class_df = (
    filtered_df
    .withColumn(
        "ft_class",
        F.when(F.col("freeze_thaw_cycles") < 40, "Low")
         .when(F.col("freeze_thaw_cycles") < 80, "Moderate")
         .otherwise("High")
    )
    .select("bridge_age", "ft_class")
    .filter(F.col("bridge_age").isNotNull())
)

# 2. Convert to pandas
age_ft_pdf = ft_class_df.toPandas()

# 3. Boxplot in matplotlib / seaborn
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=age_ft_pdf,
    x="ft_class",
    y="bridge_age",
    order=["Low", "High", "Moderate"],   # match label order in your figure
    palette=["#8da0cb", "#fc8d62", "#66c2a5"]
)

plt.title("Bridge Age Distribution by Freeze–Thaw Class", fontsize=16)
plt.xlabel("Freeze–Thaw Class", fontsize=12)
plt.ylabel("Bridge Age (Years)", fontsize=12)

plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [0]:
# -------------------------------------------------------
# STEP A — Convert Spark DF to Pandas
# -------------------------------------------------------

pdf = filtered_df.toPandas()
len(pdf)

# -------------------------------------------------------
# STEP B — Save to CSV on DBFS
# -------------------------------------------------------

#output_path = "/dbfs/FileStore/bridge_clean.csv"
# pdf.to_csv(output_path, index=False)

#print("Saved to:", output_path)



# -------------------------------------------------------
# Display Pandas DF and download manually
# -------------------------------------------------------

pdf = filtered_df.toPandas()
display(pdf)


In [0]:
# ============================================================
# PREP STEP – BUILD encoded_df FROM filtered_df  (RUN ONCE)
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# filtered_df must already exist from your cleaning pipeline and
# contain these columns:
# deck_condition_rating, freeze_thaw_cycles, bridge_age,
# adt, deck_area_sqft, state_name, main_span_material

# 1) Select needed columns
model_df = filtered_df.select(
    "deck_condition_rating",
    "freeze_thaw_cycles",
    "bridge_age",
    "adt",
    "deck_area_sqft",
    "state_name",
    "main_span_material"
)

# 2) Manual STATE encoding
state_w = Window.orderBy("state_name")
state_map = (
    model_df
    .select("state_name")
    .distinct()
    .withColumn("state_index", F.row_number().over(state_w) - 1)
)

df1 = model_df.join(state_map, on="state_name", how="left")

# 3) Manual MATERIAL encoding
mat_w = Window.orderBy("main_span_material")
material_map = (
    df1
    .select("main_span_material")
    .distinct()
    .withColumn("material_index", F.row_number().over(mat_w) - 1)
)

encoded_df = df1.join(material_map, on="main_span_material", how="left")

# Optional sanity check
encoded_df.select(
    "state_name", "state_index",
    "main_span_material", "material_index"
).show(10, truncate=False)


# ============================================================
# BINARY LOGISTIC REGRESSION: POOR vs NOT POOR (RQ1 add-on)
# ============================================================
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# 1) CREATE BINARY LABEL
# label = 1 if deck_condition_rating <= 4 (poor), else 0
bin_df = encoded_df.withColumn(
    "label",
    F.when(F.col("deck_condition_rating") <= 4, 1).otherwise(0)
)

# 2) FEATURE ENGINEERING (same pattern as improved models)
enhanced_cls_df = (
    bin_df
    .withColumn("log_adt", F.log1p(F.col("adt")))
    .withColumn("age_sq", F.col("bridge_age") ** 2)
    .withColumn("ft_sq", F.col("freeze_thaw_cycles") ** 2)
    .withColumn(
        "ft_per_age",
        F.when(F.col("bridge_age") > 0,
               F.col("freeze_thaw_cycles") / F.col("bridge_age"))
         .otherwise(0.0)
    )
    .withColumn(
        "ft_zone",
        F.when(F.col("freeze_thaw_cycles") < 20, 0)
         .when(F.col("freeze_thaw_cycles") < 50, 1)
         .when(F.col("freeze_thaw_cycles") < 80, 2)
         .otherwise(3)
    )
)

feature_cols_cls = [
    "freeze_thaw_cycles",
    "bridge_age",
    "adt",
    "deck_area_sqft",
    "state_index",
    "material_index",
    "log_adt",
    "age_sq",
    "ft_sq",
    "ft_per_age",
    "ft_zone"
]

assembler_cls = VectorAssembler(
    inputCols=feature_cols_cls,
    outputCol="features"
)

final_cls_df = assembler_cls.transform(enhanced_cls_df).select("features", "label")

# 3) TRAIN / TEST SPLIT
train_cls_df, test_cls_df = final_cls_df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_cls_df.count())
print("Testing rows :", test_cls_df.count())

# 4) LOGISTIC REGRESSION MODEL
lr_cls = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    regParam=0.1,
    elasticNetParam=0.0
)

lr_cls_model = lr_cls.fit(train_cls_df)
lr_cls_preds = lr_cls_model.transform(test_cls_df)

# 5) EVALUATION METRICS
binary_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = binary_eval.evaluate(lr_cls_preds)

multi_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

accuracy  = multi_eval.evaluate(lr_cls_preds, {multi_eval.metricName: "accuracy"})
precision = multi_eval.evaluate(lr_cls_preds, {multi_eval.metricName: "precisionByLabel"})
recall    = multi_eval.evaluate(lr_cls_preds, {multi_eval.metricName: "recallByLabel"})
f1        = multi_eval.evaluate(lr_cls_preds, {multi_eval.metricName: "f1"})

print("\n===== Logistic Regression (Poor vs Not Poor) =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("AUC (ROC):", auc)

# 6) CONFUSION MATRIX (for your report)
confusion = (
    lr_cls_preds.groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

print("\n===== Confusion Matrix =====")
display(confusion)


In [0]:
# -------------------------------------------------------
# STEP 8.1 — SELECT MODELING COLUMNS
# -------------------------------------------------------

model_df = filtered_df.select(
    "deck_condition_rating",       # Label
    "freeze_thaw_cycles",
    "bridge_age",
    "adt",
    "deck_area_sqft",
    "state_name",
    "main_span_material",
    "main_span_design"
)

model_df.show(5)
model_df.printSchema()
print("Total rows for modeling:", model_df.count())


In [0]:
# -------------------------------------------------------
# STEP 8.2 — STRING INDEXING (Categorical Encoding)
# -------------------------------------------------------

from pyspark.sql import functions as F

model_df.select("state_name").distinct().count(), \
model_df.select("main_span_material").distinct().count(), \
model_df.select("main_span_design").distinct().count()


# -------------------------------------------------------
# STEP 8.2 — MANUAL ENCODING FOR CATEGORICAL FEATURES
# (No StringIndexer, avoids MODEL_SIZE_OVERFLOW)
# -------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Start from model_df (numeric + categorical + label)
# model_df = filtered_df.select(... )  # you already created this earlier

# 1) Index states
state_window = Window.orderBy("state_name")

state_map = (
    model_df
    .select("state_name")
    .distinct()
    .withColumn("state_index", F.row_number().over(state_window) - 1)
)

df1 = model_df.join(state_map, on="state_name", how="left")

# 2) Index materials
mat_window = Window.orderBy("main_span_material")

material_map = (
    df1
    .select("main_span_material")
    .distinct()
    .withColumn("material_index", F.row_number().over(mat_window) - 1)
)

df2 = df1.join(material_map, on="main_span_material", how="left")

# (Optional) If you REALLY want design_index later, we can do the same trick,
# but let's keep it simple and stable for now.

encoded_df = df2

encoded_df.select(
    "state_name", "state_index",
    "main_span_material", "material_index"
).show(10, truncate=False)


In [0]:
%pip install xgboost==1.7.6

In [0]:
# ==============================================================
# Local XGBoost Regressor using Pandas (works on serverless)
# Target: deck_condition_rating
# ==============================================================

from pyspark.sql import functions as F
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

try:
    import xgboost as xgb
except ImportError:
    raise ImportError("XGBoost is not installed. Run in a separate cell: %pip install xgboost scikit-learn")

print(">>> Building local XGBoost model (Pandas-based)...")

# 1. OUTLIER FILTERING
base_df = encoded_df.filter(
    (F.col("deck_area_sqft") < 200000) &
    (F.col("adt") < 200000) &
    (F.col("bridge_age") > 0) &
    (F.col("bridge_age") < 200) &
    (F.col("freeze_thaw_cycles") >= 0) &
    (F.col("freeze_thaw_cycles") < 120)
)

print("Rows after outlier filtering:", base_df.count())

# 2. FEATURE ENGINEERING
feat_df = (
    base_df
    .withColumn("log_adt", F.log1p(F.col("adt")))
    .withColumn("age_sq", F.col("bridge_age")**2)
    .withColumn("ft_sq", F.col("freeze_thaw_cycles")**2)
    .withColumn(
        "ft_per_age",
        F.col("freeze_thaw_cycles") / F.col("bridge_age")
    )
    .withColumn(
        "ft_zone",
        F.when(F.col("freeze_thaw_cycles") < 20, 0)
         .when(F.col("freeze_thaw_cycles") < 50, 1)
         .when(F.col("freeze_thaw_cycles") < 80, 2)
         .otherwise(3)
    )
)

pdf = feat_df.select(
    "freeze_thaw_cycles",
    "bridge_age",
    "adt",
    "deck_area_sqft",
    "state_index",
    "material_index",
    "log_adt",
    "age_sq",
    "ft_sq",
    "ft_per_age",
    "ft_zone",
    "deck_condition_rating"
).toPandas()

print("Pandas DataFrame shape:", pdf.shape)

# 3. TRAIN / TEST SPLIT
X = pdf.drop(columns=["deck_condition_rating"])
y = pdf["deck_condition_rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

# 4. XGBOOST REGRESSOR
xgb_model = xgb.XGBRegressor(
    max_depth=10,
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    reg_alpha=0.0,
    min_child_weight=1,
    objective="reg:squarederror",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

print(">>> Training XGBoost...")
xgb_model.fit(X_train, y_train)

# 5. EVALUATION  🔧 (FIXED RMSE CALCULATION)
y_pred = xgb_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))  # ← no 'squared' argument
r2   = r2_score(y_test, y_pred)

print("\n===== XGBoost Regression Results (Local) =====")
print("RMSE:", rmse)
print("R²  :", r2)

# 6. FEATURE IMPORTANCE
fi = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n===== XGBoost Feature Importance =====")
print(fi)

In [0]:
# ======================================================
# RQ1 – Predictive Modeling: Linear Regression vs GBT
# Using SAME pipeline as final model (manual z-scores)
# ======================================================

from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

print(">>> RQ1 – Predictive Modeling Started")

# -------------------------------------------------------
# 1. REMOVE OUTLIERS
# -------------------------------------------------------
clean_df = encoded_df.filter(
    (F.col("deck_area_sqft") < 200000) &
    (F.col("adt") < 200000) &
    (F.col("bridge_age") > 0) &
    (F.col("bridge_age") < 200) &
    (F.col("freeze_thaw_cycles") >= 0) &
    (F.col("freeze_thaw_cycles") < 120)
)

print("Rows after outlier filtering:", clean_df.count())

# -------------------------------------------------------
# 2. FEATURE ENGINEERING
# -------------------------------------------------------
enhanced_df = (
    clean_df
    .withColumn("log_adt", F.log1p(F.col("adt")))
    .withColumn("age_sq", F.col("bridge_age") ** 2)
    .withColumn("ft_sq", F.col("freeze_thaw_cycles") ** 2)
    .withColumn("ft_per_age",
        F.col("freeze_thaw_cycles") / F.col("bridge_age"))
    .withColumn("ft_zone",
        F.when(F.col("freeze_thaw_cycles") < 20, 0)
         .when(F.col("freeze_thaw_cycles") < 50, 1)
         .when(F.col("freeze_thaw_cycles") < 80, 2)
         .otherwise(3)
    )
)

# All numeric features to standardize
num_cols = [
    "freeze_thaw_cycles",
    "bridge_age",
    "adt",
    "deck_area_sqft",
    "log_adt",
    "age_sq",
    "ft_sq",
    "ft_per_age"
]

# -------------------------------------------------------
# 3. MANUAL STANDARDIZATION
# -------------------------------------------------------
stats = enhanced_df.select(
    *[F.mean(c).alias(f"{c}_mean") for c in num_cols],
    *[F.stddev(c).alias(f"{c}_std") for c in num_cols]
).collect()[0]

standardized_df = enhanced_df

for c in num_cols:
    mean = stats[f"{c}_mean"]
    std  = stats[f"{c}_std"] if stats[f"{c}_std"] != 0 else 1.0
    standardized_df = standardized_df.withColumn(
        f"{c}_z",
        (F.col(c) - F.lit(mean)) / F.lit(std)
    )

# Final ML columns for RQ1
feature_cols = [
    "freeze_thaw_cycles_z",
    "bridge_age_z",
    "adt_z",
    "deck_area_sqft_z",
    "log_adt_z",
    "age_sq_z",
    "ft_sq_z",
    "ft_per_age_z",
    "ft_zone",
    "state_index",
    "material_index"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

final_df = assembler.transform(standardized_df) \
    .select("features", "deck_condition_rating")

# -------------------------------------------------------
# 4. TRAIN / TEST SPLIT
# -------------------------------------------------------
train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)

print("Training:", train_df.count(), " Testing:", test_df.count())

evaluator_rmse = RegressionEvaluator(
    labelCol="deck_condition_rating",
    predictionCol="prediction",
    metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="deck_condition_rating",
    predictionCol="prediction",
    metricName="r2"
)

# -------------------------------------------------------
# 5. LINEAR REGRESSION FOR RQ1
# -------------------------------------------------------
print("\n>>> Training Linear Regression for RQ1...")
lr = LinearRegression(
    featuresCol="features",
    labelCol="deck_condition_rating",
    maxIter=100,
    regParam=0.05,
    elasticNetParam=0.0
)

lr_model = lr.fit(train_df)
lr_pred = lr_model.transform(test_df)

lr_rmse = evaluator_rmse.evaluate(lr_pred)
lr_r2   = evaluator_r2.evaluate(lr_pred)

print("\n===== RQ1: Linear Regression =====")
print("RMSE:", lr_rmse)
print("R2  :", lr_r2)

# -------------------------------------------------------
# 6. GRADIENT-BOOSTED TREES FOR RQ1
# -------------------------------------------------------
print("\n>>> Training GBT for RQ1...")
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="deck_condition_rating",
    maxDepth=7,
    maxIter=50,
    stepSize=0.05,
    seed=42
)

gbt_model = gbt.fit(train_df)
gbt_pred = gbt_model.transform(test_df)

gbt_rmse = evaluator_rmse.evaluate(gbt_pred)
gbt_r2   = evaluator_r2.evaluate(gbt_pred)

print("\n===== RQ1: Gradient-Boosted Trees (GBT) =====")
print("GBT RMSE:", gbt_rmse)
print("GBT R2  :", gbt_r2)

# -------------------------------------------------------
# 7. FEATURE IMPORTANCE FOR RQ1 REPORT
# -------------------------------------------------------
import pandas as pd

feat_imp = list(zip(feature_cols, gbt_model.featureImportances))
feat_imp_df = pd.DataFrame(feat_imp, columns=["feature", "importance"]) \
                    .sort_values("importance", ascending=False)

print("\n===== RQ1: GBT Feature Importance =====")
display(feat_imp_df)

print("\n>>> RQ1 Completed.")


In [0]:
# ============================================================
# RQ2 – Environmental Analytics:
# Cold vs Temperate states based on freeze–thaw cycles
# Comparing deterioration rate of deck_condition_rating vs age
# Requires: filtered_df (from your cleaned dataset)
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

print(">>> RQ2 – Starting from filtered_df")

# ------------------------------------------------------------
# 1. FILTER rows with required columns present
# ------------------------------------------------------------
rq2_df = filtered_df.filter(
    F.col("deck_condition_rating").isNotNull() &
    F.col("freeze_thaw_cycles").isNotNull() &
    F.col("bridge_age").isNotNull()
)

print("Rows available for RQ2:", rq2_df.count())
rq2_df.select(
    "state_name", "bridge_age", "freeze_thaw_cycles", "deck_condition_rating"
).show(5, truncate=False)

# ------------------------------------------------------------
# 2. COMPUTE avg freeze–thaw cycles per state
# ------------------------------------------------------------
state_ft = (
    rq2_df
    .groupBy("state_name")
    .agg(F.avg("freeze_thaw_cycles").alias("avg_ft"))
)

print("\n>>> Average freeze–thaw cycles per state:")
state_ft.orderBy(F.desc("avg_ft")).show(10, truncate=False)

# ------------------------------------------------------------
# 3. DEFINE CLIMATE ZONES (Cold vs Temperate)
#    You can adjust threshold if needed.
#    Example: Cold if avg_ft >= 70, else Temperate
# ------------------------------------------------------------
state_zones = (
    state_ft
    .withColumn(
        "climate_zone",
        F.when(F.col("avg_ft") >= 70, F.lit("Cold"))
         .otherwise(F.lit("Temperate"))
    )
)

print("\n>>> State climate zones (Cold / Temperate):")
state_zones.show(20, truncate=False)

# JOIN zones back to main DF
zone_df = rq2_df.join(
    state_zones.select("state_name", "climate_zone"),
    on="state_name",
    how="left"
)

zone_df.select(
    "state_name", "climate_zone", "bridge_age",
    "freeze_thaw_cycles", "deck_condition_rating"
).show(10, truncate=False)

# ------------------------------------------------------------
# 4. AGGREGATE: avg deck rating vs age bucket per climate zone
# ------------------------------------------------------------
# Bucket age into 5-year bins for smoother curves
zone_df_age = zone_df.withColumn(
    "age_bucket",
    (F.col("bridge_age") / 5).cast("int") * 5
)

avg_by_zone_age = (
    zone_df_age
    .groupBy("climate_zone", "age_bucket")
    .agg(
        F.avg("deck_condition_rating").alias("avg_rating"),
        F.count("*").alias("n_bridges")
    )
    .orderBy("climate_zone", "age_bucket")
)

print("\n>>> Average deck rating by age bucket and climate zone:")
display(avg_by_zone_age)

# In Databricks, you can now:
# - Click "Plot" on avg_by_zone_age
# - x-axis: age_bucket
# - y-axis: avg_rating
# - series group: climate_zone
# to visually compare deterioration curves.

# ------------------------------------------------------------
# 5. LINEAR REGRESSION: rating ~ age, per climate zone
#    This gives a "deterioration slope" for Cold vs Temperate
# ------------------------------------------------------------
from pyspark.ml.feature import VectorAssembler

# Prepare separate DFs for each zone
cold_df = zone_df.filter(F.col("climate_zone") == "Cold") \
                 .select(
                     F.col("bridge_age").alias("age"),
                     F.col("deck_condition_rating").alias("label")
                 )

temp_df = zone_df.filter(F.col("climate_zone") == "Temperate") \
                 .select(
                     F.col("bridge_age").alias("age"),
                     F.col("deck_condition_rating").alias("label")
                 )

print("\nCold rows:", cold_df.count(), " Temperate rows:", temp_df.count())

assembler = VectorAssembler(inputCols=["age"], outputCol="features")

cold_m = assembler.transform(cold_df).select("features", "label")
temp_m = assembler.transform(temp_df).select("features", "label")

# Train simple linear regression in each zone
lr_cold = LinearRegression(featuresCol="features", labelCol="label")
lr_temp = LinearRegression(featuresCol="features", labelCol="label")

cold_model = lr_cold.fit(cold_m)
temp_model = lr_temp.fit(temp_m)

# Evaluate R² for each
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)

cold_pred = cold_model.transform(cold_m)
temp_pred = temp_model.transform(temp_m)

cold_r2 = evaluator.evaluate(cold_pred)
temp_r2 = evaluator.evaluate(temp_pred)

print("\n===== RQ2 – Deterioration Slope (rating vs age) =====")
print("Cold zone – slope (coef on age):", float(cold_model.coefficients[0]))
print("Cold zone – intercept          :", float(cold_model.intercept))
print("Cold zone – R²                 :", cold_r2)

print("\nTemperate zone – slope (coef):  ", float(temp_model.coefficients[0]))
print("Temperate zone – intercept      :", float(temp_model.intercept))
print("Temperate zone – R²             :", temp_r2)

print("\n>>> Interpretation hint:")
print("More negative slope = faster deterioration with age.")
print("Compare Cold vs Temperate slopes to answer RQ2.")


In [0]:
# ============================================================
# RQ3 – Material & Design Influence on Freeze–Thaw Deterioration
# Goal: See if material/design moderate the effect of freeze–thaw
# Uses: filtered_df  (same as RQ1/RQ2)
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd

print(">>> RQ3 – Starting from filtered_df")

# ------------------------------------------------------------
# 1. Base subset: require all key columns present
# ------------------------------------------------------------
base_df = filtered_df.filter(
    F.col("deck_condition_rating").isNotNull() &
    F.col("freeze_thaw_cycles").isNotNull() &
    F.col("bridge_age").isNotNull() &
    F.col("adt").isNotNull() &
    F.col("deck_area_sqft").isNotNull() &
    F.col("main_span_material").isNotNull() &
    F.col("main_span_design").isNotNull() &
    F.col("state_name").isNotNull()
)

print("Rows available for RQ3:", base_df.count())
base_df.select(
    "state_name", "main_span_material", "main_span_design",
    "bridge_age", "freeze_thaw_cycles", "deck_condition_rating"
).show(5, truncate=False)

# ------------------------------------------------------------
# 2. Manual categorical encoding (state, material, design)
#    (Avoid StringIndexer to prevent MODEL_SIZE_OVERFLOW)
# ------------------------------------------------------------
state_w = Window.orderBy("state_name")
state_map = (
    base_df
    .select("state_name").distinct()
    .withColumn("state_index", F.row_number().over(state_w) - 1)
)

mat_w = Window.orderBy("main_span_material")
material_map = (
    base_df
    .select("main_span_material").distinct()
    .withColumn("material_index", F.row_number().over(mat_w) - 1)
)

des_w = Window.orderBy("main_span_design")
design_map = (
    base_df
    .select("main_span_design").distinct()
    .withColumn("design_index", F.row_number().over(des_w) - 1)
)

df_idx = (
    base_df
    .join(state_map, on="state_name", how="left")
    .join(material_map, on="main_span_material", how="left")
    .join(design_map, on="main_span_design", how="left")
)

df_idx.select(
    "state_name", "state_index",
    "main_span_material", "material_index",
    "main_span_design", "design_index"
).show(10, truncate=False)

# ------------------------------------------------------------
# 3. Feature engineering + interaction terms
# ------------------------------------------------------------
feat_df = (
    df_idx
    # core engineered numeric features
    .withColumn("log_adt", F.log1p(F.col("adt")))
    .withColumn("age_sq", F.col("bridge_age") ** 2)
    .withColumn("ft_sq", F.col("freeze_thaw_cycles") ** 2)
    .withColumn(
        "ft_per_age",
        F.when(F.col("bridge_age") > 0,
               F.col("freeze_thaw_cycles") / F.col("bridge_age"))
         .otherwise(0.0)
    )
    .withColumn(
        "ft_zone",
        F.when(F.col("freeze_thaw_cycles") < 20, 0)
         .when(F.col("freeze_thaw_cycles") < 50, 1)
         .when(F.col("freeze_thaw_cycles") < 80, 2)
         .otherwise(3)
    )
    # interaction terms: moderation of freeze–thaw by material & design
    .withColumn("ft_x_material", F.col("freeze_thaw_cycles") * F.col("material_index"))
    .withColumn("ft_x_design",   F.col("freeze_thaw_cycles") * F.col("design_index"))
)

print("\n>>> Example rows with interaction features:")
feat_df.select(
    "freeze_thaw_cycles", "material_index", "design_index",
    "ft_x_material", "ft_x_design", "deck_condition_rating"
).show(10, truncate=False)

# ------------------------------------------------------------
# 4. Assemble features for GBT model
# ------------------------------------------------------------
feature_cols = [
    # base environment + age
    "freeze_thaw_cycles",
    "bridge_age",
    "adt",
    "deck_area_sqft",
    # categorical indices
    "state_index",
    "material_index",
    "design_index",
    # engineered numeric
    "log_adt",
    "age_sq",
    "ft_sq",
    "ft_per_age",
    "ft_zone",
    # interaction (moderation) terms
    "ft_x_material",
    "ft_x_design"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

model_df = assembler.transform(feat_df).select(
    "features", "deck_condition_rating"
)

train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print("Train rows:", train_df.count(), " Test rows:", test_df.count())

# ------------------------------------------------------------
# 5. Train GBTRegressor for RQ3
# ------------------------------------------------------------
evaluator_rmse = RegressionEvaluator(
    labelCol="deck_condition_rating",
    predictionCol="prediction",
    metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="deck_condition_rating",
    predictionCol="prediction",
    metricName="r2"
)

print("\n>>> Training GBTRegressor for RQ3...")
gbt = GBTRegressor(
    labelCol="deck_condition_rating",
    featuresCol="features",
    maxDepth=7,
    maxIter=50,
    stepSize=0.05,
    seed=42
)

gbt_model = gbt.fit(train_df)
gbt_pred  = gbt_model.transform(test_df)

rq3_rmse = evaluator_rmse.evaluate(gbt_pred)
rq3_r2   = evaluator_r2.evaluate(gbt_pred)

print("\n===== RQ3 – GBT Model Performance =====")
print("RMSE:", rq3_rmse)
print("R²  :", rq3_r2)

# ------------------------------------------------------------
# 6. Feature importance – focus on interactions
# ------------------------------------------------------------
fi = pd.DataFrame(
    list(zip(feature_cols, gbt_model.featureImportances)),
    columns=["feature", "importance"]
).sort_values("importance", ascending=False)

print("\n===== RQ3 – GBT Feature Importance =====")
display(fi)

# ------------------------------------------------------------
# 7. Extra EDA for RQ3 (for tables/plots in paper)
# ------------------------------------------------------------

print("\n>>> Average rating & freeze–thaw by MATERIAL:")
mat_summary = (
    df_idx
    .groupBy("main_span_material")
    .agg(
        F.count("*").alias("n_bridges"),
        F.avg("deck_condition_rating").alias("avg_rating"),
        F.avg("freeze_thaw_cycles").alias("avg_ft")
    )
    .orderBy("avg_rating")
)
display(mat_summary)

print("\n>>> Average rating & freeze–thaw by DESIGN:")
design_summary = (
    df_idx
    .groupBy("main_span_design")
    .agg(
        F.count("*").alias("n_bridges"),
        F.avg("deck_condition_rating").alias("avg_rating"),
        F.avg("freeze_thaw_cycles").alias("avg_ft")
    )
    .orderBy("avg_rating")
)
display(design_summary)

print("\n>>> RQ3 modeling + EDA completed.")
